## 1) Setup & Installs

## 2) Configuration

In [ ]:
# 5,3=context, 50----reranker---5-8


# bi encoder---q , c

In [1]:
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
CROSS_ENCODER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
GEN_REWRITE_MODEL = "google/flan-t5-base"

ENABLE_QUERY_REWRITING = True
ENABLE_RERANKING = True

TOP_K = 5
RERANK_K = 3

print("Config set.")


Config set.


## 3) Build a Small Demo Corpus

In [2]:

CORPUS = [
    {"id": "1", "text": "Retrieval-Augmented Generation (RAG) enriches prompts with retrieved evidence to improve factuality."},
    {"id": "2", "text": "Query rewriting clarifies intent by expanding acronyms, resolving pronouns, and adding synonyms."},
    {"id": "3", "text": "Cross-encoders re-rank candidates by scoring query-passage pairs with a joint encoder."},
    {"id": "4", "text": "Groundedness measures whether the answer is supported by the provided context."},
    {"id": "5", "text": "LangGraph enables building multi-actor agent workflows with control over LLM steps."},
    {"id": "6", "text": "Dense retrieval with MiniLM or BGE embeddings is fast and effective for small corpora."},
    {"id": "7", "text": "RAGAS provides automatic metrics like faithfulness, answer relevancy, context precision and recall."},
    {"id": "8", "text": "Ablations help compare strategies like query rewriting and re-ranking to quantify improvements."}
]



print(f"Loaded {len(CORPUS)} passages.")


Loaded 8 passages.


## 4) Embeddings, FAISS Retrieval, Cross-Encoder Re-Ranking

In [5]:
# !pip install faiss-cpu

In [6]:

from typing import List, Dict, Any
import numpy as np

_embed_model = None
def get_embeddings(texts: List[str]) -> np.ndarray:
    global _embed_model
    if _embed_model is None:
        from sentence_transformers import SentenceTransformer
        _embed_model = SentenceTransformer(EMBED_MODEL)
    return _embed_model.encode(texts, normalize_embeddings=True, convert_to_numpy=True)


def build_faiss(corpus: List[Dict[str, str]]):
    import faiss
    texts = [c["text"] for c in corpus]
    ids = [c["id"] for c in corpus]
    embs = get_embeddings(texts).astype("float32")
    index = faiss.IndexFlatIP(embs.shape[1])
    index.add(embs)
    return index, ids, texts, embs


def search(index, query: str, ids, texts, embs, top_k=5):
    q_emb = get_embeddings([query]).astype("float32")
    D, I = index.search(q_emb, top_k)
    hits = []
    for score, idx in zip(D[0], I[0]):
        if idx == -1:
            continue
        hits.append({"id": ids[idx], "text": texts[idx], "score": float(score)})
    return hits

_cross_encoder = None
def rerank(query: str, hits: List[Dict[str, Any]], top_k=3):
    global _cross_encoder
    if _cross_encoder is None:
        from sentence_transformers import CrossEncoder
        _cross_encoder = CrossEncoder(CROSS_ENCODER_MODEL)
    pairs = [(query, h["text"]) for h in hits] #c+q
    scores = _cross_encoder.predict(pairs)
    for h, s in zip(hits, scores):
        h["rerank_score"] = float(s)
    return sorted(hits, key=lambda x: x["rerank_score"], reverse=True)[:top_k]


index, IDS, TEXTS, EMBEDS = build_faiss(CORPUS)
print("FAISS index ready.")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS index ready.


## 5) Lightweight Generator: FLAN-T5 for Query Rewriting & Answers

In [8]:

from transformers import pipeline

t5 = pipeline("text-generation", model=GEN_REWRITE_MODEL)

REWRITE_SYSTEM = "Rewrite the user query for better retrieval: expand acronyms, clarify pronouns, add synonyms, keep intent."
def rewrite_query(q: str) -> str:
    if not ENABLE_QUERY_REWRITING:
        return q
    prompt = f"{REWRITE_SYSTEM}\n\nUser query: {q}\nRewritten (concise): "
    out = t5(prompt, max_new_tokens=64, do_sample=False)
    return out[0]["generated_text"].strip()

ANSWER_SYSTEM = "You are a helpful assistant. Answer strictly using ONLY the provided context. If insufficient, say you don't know."
def answer_with_context(q: str, ctx_list: List[Dict[str, Any]]) -> str:
    ctx = "\n\n".join([f"- {c['text']}" for c in ctx_list])
    prompt = f"{ANSWER_SYSTEM}\n\nQuestion: {q}\n\nContext:\n{ctx}\n\nAnswer:"
    out = t5(prompt, max_new_tokens=256, do_sample=False)
    return out[0]["generated_text"].strip()


model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCausalLM', 'FalconForCausalLM', 'FalconH1ForCausalLM', 'FalconMambaForCausa

## 6) RAG Pipeline: Rewrite → Retrieve → Re-Rank → Answer

In [9]:

def rag_pipeline(q: str, top_k=5, rerank_k=3) -> dict:
    q_rw = rewrite_query(q)
    hits = search(index, q_rw, IDS, TEXTS, EMBEDS, top_k=top_k)
    if ENABLE_RERANKING and hits:
        hits_rr = rerank(q_rw, hits, top_k=rerank_k)
    else:
        hits_rr = hits[:rerank_k]
    ans = answer_with_context(q, hits_rr)
    return {"query": q, "query_rewritten": q_rw, "retrieved": hits, "reranked": hits_rr, "answer": ans}


In [10]:
DEMO_QUERIES = [
    "How does RAG reduce hallucinations?",
    "Why is cross-encoder reranking useful?",
    "What is query rewriting and how does it help retrieval?"
]

In [11]:
for q in DEMO_QUERIES:
    out = rag_pipeline(q)
    print("="*80)
    print("Q:", out["query"])
    print("Rewritten:", out["query_rewritten"])
    print("Top ctx (reranked):")
    for c in out["reranked"]:
        print(f"  • ({c.get('rerank_score', c['score']):.3f}) {c['text']}")
    print("\nAnswer:", out["answer"][:300])

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

Q: How does RAG reduce hallucinations?
Rewritten: Rewrite the user query for better retrieval: expand acronyms, clarify pronouns, add synonyms, keep intent.

User query: How does RAG reduce hallucinations?
Rewritten (concise):
Top ctx (reranked):
  • (2.969) Query rewriting clarifies intent by expanding acronyms, resolving pronouns, and adding synonyms.
  • (-4.344) Retrieval-Augmented Generation (RAG) enriches prompts with retrieved evidence to improve factuality.
  • (-6.845) Ablations help compare strategies like query rewriting and re-ranking to quantify improvements.

Answer: You are a helpful assistant. Answer strictly using ONLY the provided context. If insufficient, say you don't know.

Question: How does RAG reduce hallucinations?

Context:
- Query rewriting clarifies intent by expanding acronyms, resolving pronouns, and adding synonyms.

- Retrieval-Augmented Genera


Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Why is cross-encoder reranking useful?
Rewritten: Rewrite the user query for better retrieval: expand acronyms, clarify pronouns, add synonyms, keep intent.

User query: Why is cross-encoder reranking useful?
Rewritten (concise):
Top ctx (reranked):
  • (3.777) Query rewriting clarifies intent by expanding acronyms, resolving pronouns, and adding synonyms.
  • (-5.466) Ablations help compare strategies like query rewriting and re-ranking to quantify improvements.
  • (-5.548) Cross-encoders re-rank candidates by scoring query-passage pairs with a joint encoder.

Answer: You are a helpful assistant. Answer strictly using ONLY the provided context. If insufficient, say you don't know.

Question: Why is cross-encoder reranking useful?

Context:
- Query rewriting clarifies intent by expanding acronyms, resolving pronouns, and adding synonyms.

- Ablations help compare 


Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What is query rewriting and how does it help retrieval?
Rewritten: Rewrite the user query for better retrieval: expand acronyms, clarify pronouns, add synonyms, keep intent.

User query: What is query rewriting and how does it help retrieval?
Rewritten (concise):
Top ctx (reranked):
  • (5.036) Query rewriting clarifies intent by expanding acronyms, resolving pronouns, and adding synonyms.
  • (-4.332) Ablations help compare strategies like query rewriting and re-ranking to quantify improvements.
  • (-4.937) Retrieval-Augmented Generation (RAG) enriches prompts with retrieved evidence to improve factuality.

Answer: You are a helpful assistant. Answer strictly using ONLY the provided context. If insufficient, say you don't know.

Question: What is query rewriting and how does it help retrieval?

Context:
- Query rewriting clarifies intent by expanding acronyms, resolving pronouns, and adding synonyms.

- Ablati


## 7) Evaluation: RAG Triad (Heuristics) + Local Judge (FLAN-T5)

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re, json, pandas as pd

def cosine_sim(a: str, b: str) -> float:
    vec = TfidfVectorizer().fit([a, b])
    A = vec.transform([a])
    B = vec.transform([b])
    return float(cosine_similarity(A, B)[0][0])

def triad_metrics(query: str, answer: str, ctx_list: list) -> dict:
    ctx_text = "\n".join([c["text"] for c in ctx_list]) if ctx_list else ""
    context_relevance = cosine_sim(query, ctx_text) if ctx_text else 0.0
    answer_relevance = cosine_sim(query, answer) if answer else 0.0
    def tokens(s):
        return set(re.findall(r"\w+", s.lower()))
    ans_t = tokens(answer)
    ctx_t = tokens(ctx_text)
    groundedness = len(ans_t & ctx_t) / (len(ans_t) + 1e-6)
    return {
        "context_relevance": round(context_relevance, 3),
        "answer_relevance": round(answer_relevance, 3),
        "groundedness_overlap": round(groundedness, 3),
    }

JUDGE_TMPL = """Question: {q}

Context:
{ctx}

Assistant Answer:
{a}

Rate each 0-5 with a short reason:
1) Groundedness (supported by context)
2) Context Relevance (is the retrieved context relevant?)
3) Answer Relevance (does the answer address the question?)

Return JSON with keys: groundedness, context_relevance, answer_relevance, reasons.
"""

def local_judge(query: str, answer: str, ctx_list: list):
    ctx = "\n".join([f"- {c['text']}" for c in ctx_list]) if ctx_list else "(no context)"
    prompt = JUDGE_TMPL.format(q=query, a=answer, ctx=ctx)
    out = t5(prompt, max_new_tokens=256, do_sample=False)[0]["generated_text"].strip()
    try:
        start, end = out.find("{"), out.rfind("}")
        return json.loads(out[start:end+1])
    except Exception:
        return {"raw": out}

rows = []
for q in DEMO_QUERIES:
    o = rag_pipeline(q)
    tri = triad_metrics(o["query"], o["answer"], o["reranked"])
    judge = local_judge(o["query"], o["answer"], o["reranked"])
    rows.append({
        "query": o["query"],
        "rewritten": o["query_rewritten"],
        **{f"triad_{k}": v for k, v in tri.items()},
        "judge": judge
    })

df_eval = pd.DataFrame(rows)

df_eval

Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `

,query,rewritten,triad_context_relevance,triad_answer_relevance,triad_groundedness_overlap,judge
0,How does RAG reduce hallucinations?,Rewrite the user query for better retrieval: e...,0.036,0.222,0.589,{'raw': 'Question: How does RAG reduce halluci...
1,Why is cross-encoder reranking useful?,Rewrite the user query for better retrieval: e...,0.063,0.257,0.600,{'raw': 'Question: Why is cross-encoder rerank...
2,What is query rewriting and how does it help r...,Rewrite the user query for better retrieval: e...,0.239,0.461,0.579,{'raw': 'Question: What is query rewriting and...


## 8) RAGAS Metrics (Open-Source)

In [ ]:
try:
    from ragas import evaluate
    from ragas.metrics import (faithfulness, answer_relevancy, context_precision, context_recall, answer_similarity)
    from datasets import Dataset
    import pandas as pd

    eval_rows = []
    for q in DEMO_QUERIES:
        out = rag_pipeline(q)
        contexts = [c["text"] for c in out["reranked"]]
        eval_rows.append({
            "question": out["query"],
            "contexts": contexts,
            "answer": out["answer"],
            "ground_truth": " ".join(contexts[:1]) if contexts else ""
        })
    ragas_ds = Dataset.from_pandas(pd.DataFrame(eval_rows))

    scores = evaluate(
        ragas_ds,
        metrics=[faithfulness, answer_relevancy, context_precision, context_recall, answer_similarity],
    )
    ragas_df = scores.to_pandas()
    display(ragas_df)
    ragas_df
    print("RAGAS evaluation commented out as it requires an API key.")
except Exception as e:
    print("RAGAS evaluation failed. Ensure ragas/datasets are installed. Error:", e)

## 9) Ablation: Baseline vs Rewrite vs Rerank vs Both

In [13]:
import pandas as pd

def run_eval(rewrite: bool, rerank: bool) -> pd.DataFrame:
    global ENABLE_QUERY_REWRITING, ENABLE_RERANKING
    old_rw, old_rr = ENABLE_QUERY_REWRITING, ENABLE_RERANKING
    ENABLE_QUERY_REWRITING, ENABLE_RERANKING = rewrite, rerank

    rows = []
    for q in DEMO_QUERIES:
        o = rag_pipeline(q)
        tri = triad_metrics(o["query"], o["answer"], o["reranked"])
        rows.append({"query": q, "rewrite": rewrite, "rerank": rerank, **tri})

    ENABLE_QUERY_REWRITING, ENABLE_RERANKING = old_rw, old_rr
    return pd.DataFrame(rows)

df_base   = run_eval(False, False)
df_rw     = run_eval(True,  False)
df_rr     = run_eval(False, True)
df_both   = run_eval(True,  True)

comp = pd.concat([
    df_base.assign(setting="baseline"),
    df_rw.assign(setting="rewrite_only"),
    df_rr.assign(setting="rerank_only"),
    df_both.assign(setting="rewrite+rerank"),
], ignore_index=True)

display(comp)
# comp

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both 

,query,rewrite,rerank,context_relevance,answer_relevance,groundedness_overlap,setting
0,How does RAG reduce hallucinations?,False,False,0.037,0.213,0.618,baseline
1,Why is cross-encoder reranking useful?,False,False,0.105,0.295,0.638,baseline
2,What is query rewriting and how does it help r...,False,False,0.218,0.446,0.579,baseline
3,How does RAG reduce hallucinations?,True,False,0.036,0.222,0.589,rewrite_only
4,Why is cross-encoder reranking useful?,True,False,0.063,0.257,0.600,rewrite_only
5,What is query rewriting and how does it help r...,True,False,0.305,0.502,0.603,rewrite_only
6,How does RAG reduce hallucinations?,False,True,0.037,0.213,0.618,rerank_only
7,Why is cross-encoder reranking useful?,False,True,0.067,0.266,0.614,rerank_only
8,What is query rewriting and how does it help r...,False,True,0.239,0.461,0.579,rerank_only
9,How does RAG reduce hallucinations?,True,True,0.036,0.222,0.589,rewrite+rerank


In [ ]:
# Q---RQ---MRQ


# what about revenue last Year?

# Q1: revenue FY2025
# Q2: financial report 2025 revenue
# Q3: Annual revenue statement 2025

# Decomposition